# 🎙️ XTTS v2 Vietnamese Fine-Tuning

**Datasets cần add (Settings → Add Data):**
| Dataset | Dùng để |
|---|---|
| `tinthnhphm21022004/data-speech-to-text` | File audio WAV |
| `thanhphamtien2102224/weight-phowhisper` | Manifest JSONL |

**Settings:** Accelerator → **GPU T4 x1** | Internet → **ON**

---
⚠️ **Quan trọng:** Sau Cell 0 (pin numpy), bắt buộc **Restart Kernel** rồi chạy lại từ Cell 1.

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 0 — Fix numpy (CHẠY ĐẦU TIÊN, sau đó Restart Kernel)
#
# Kaggle Python 3.12 mặc định numpy 2.x nhưng scipy/sklearn/
# transformers được build với numpy 1.x → crash khi import TTS.
# Downgrade về 1.26.4 là bản cuối numpy 1.x, fix hoàn toàn.
# ════════════════════════════════════════════════════════════
import subprocess, sys

import numpy as np
print(f'numpy hiện tại: {np.__version__}')

if int(np.__version__.split('.')[0]) >= 2:
    print('⚠️  numpy 2.x không tương thích với scipy/sklearn trên Kaggle')
    print('🔧 Đang downgrade numpy → 1.26.4 ...')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '-q',
         'numpy==1.26.4', '--force-reinstall']
    )
    print('\n✅ Downgrade xong!')
    print('🔴 BÂY GIỜ: Run → Restart & Clear Output → rồi chạy lại từ Cell 1')
else:
    print(f'✅ numpy {np.__version__} OK, không cần downgrade')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 1 — Kiểm tra GPU + numpy
# ════════════════════════════════════════════════════════════
import torch, os, numpy as np

print(f'numpy  : {np.__version__}')   # phải là 1.26.x
print(f'PyTorch: {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️  Không có GPU!')

# Kiểm tra numpy version
if int(np.__version__.split('.')[0]) >= 2:
    raise RuntimeError('❌ numpy vẫn là 2.x! Hãy chạy Cell 0 rồi Restart Kernel')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 2 — Clone code từ GitHub
# Mỗi lần fix bug: git push → chạy lại cell này là xong
# ════════════════════════════════════════════════════════════
import sys, os, shutil, importlib

REPO_DIR = '/kaggle/working/finetuneXTTSv2'

# 1. Xoá repo cũ trên disk
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print('🗑️  Đã xoá repo cũ trên disk')

# 2. Xoá toàn bộ module xtts_finetune khỏi Python cache
#    (bắt buộc, không thì Python vẫn dùng bản cũ dù đã clone lại)
stale = [k for k in sys.modules if k.startswith('xtts_finetune')]
for k in stale:
    del sys.modules[k]
if stale:
    print(f'🗑️  Đã xoá {len(stale)} module cache: {stale}')

# 3. Clone repo mới nhất
!git clone https://github.com/thanhptks212k4/finetuneXTTSv2.git {REPO_DIR}

# 4. Thêm vào Python path + invalidate cache
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

# 5. Test import (sẽ load bản mới từ disk)
from xtts_finetune.config import TrainingConfig
from xtts_finetune.utils import get_logger, set_seed
print('\n✅ Clone thành công, import OK — code mới nhất đã load')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 3 — Cài thư viện
# ════════════════════════════════════════════════════════════
# Dùng idiap/coqui-ai-TTS (fork tương thích Python 3.12)
!pip install -q --upgrade pip setuptools wheel
!pip install -q git+https://github.com/idiap/coqui-ai-TTS.git
!pip install -q huggingface_hub librosa soundfile

# Đảm bảo numpy không bị upgrade lại
!pip install -q 'numpy==1.26.4'

# Verify
import numpy as np
from TTS.api import TTS
import TTS as _tts_pkg
print(f'✅ numpy  : {np.__version__}')
print(f'✅ TTS    : {_tts_pkg.__version__}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 4 — Đường dẫn dataset + xem manifest
# ════════════════════════════════════════════════════════════
import json

AUDIO_ROOT    = '/kaggle/input/datasets/tinthnhphm21022004/data-speech-to-text'
MANIFEST_ROOT = '/kaggle/input/datasets/thanhphamtien2102224/weight-phowhisper'
TRAIN_JSONL   = f'{MANIFEST_ROOT}/train_full_manifest.jsonl'
TEST_JSONL    = f'{MANIFEST_ROOT}/test_manifest.jsonl'

print('📂 Kiểm tra đường dẫn:')
for p in [AUDIO_ROOT, TRAIN_JSONL, TEST_JSONL]:
    status = '✅' if os.path.exists(p) else '❌ KHÔNG TÌM THẤY'
    print(f'  {status}  {p}')

print('\n📄 2 dòng đầu train manifest:')
with open(TRAIN_JSONL, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 2: break
        print(json.dumps(json.loads(line.strip()), ensure_ascii=False, indent=2))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 5 — Fix manifest path + chọn reference.wav
#
# Manifest gốc chứa path cũ: /kaggle/input/data-speech-to-text/...
# Cần rewrite sang:  /kaggle/input/datasets/tinthnhphm21022004/data-speech-to-text/...
# ════════════════════════════════════════════════════════════
import json, soundfile as sf

FIXED_TRAIN = '/kaggle/working/train_manifest.jsonl'
FIXED_TEST  = '/kaggle/working/test_manifest.jsonl'
OLD_PREFIX  = '/kaggle/input/data-speech-to-text'
NEW_PREFIX  = AUDIO_ROOT

def fix_manifest(src, dst):
    ok, missing = 0, 0
    with open(src, 'r', encoding='utf-8') as fin, \
         open(dst, 'w', encoding='utf-8') as fout:
        for line in fin:
            obj = json.loads(line.strip())
            obj['audio'] = obj['audio'].replace(OLD_PREFIX, NEW_PREFIX)
            if os.path.exists(obj['audio']):
                fout.write(json.dumps(obj, ensure_ascii=False) + '\n')
                ok += 1
            else:
                missing += 1
    print(f'  ✅ {ok:,} mẫu hợp lệ | ❌ {missing} mẫu thiếu file')

print('🔧 Fix train manifest...')
fix_manifest(TRAIN_JSONL, FIXED_TRAIN)
print('🔧 Fix test manifest...')
fix_manifest(TEST_JSONL, FIXED_TEST)

# Tìm reference.wav (file wav đầu tiên trong dataset)
REFERENCE_WAV = None
for root, dirs, files in os.walk(AUDIO_ROOT):
    for f in sorted(files):
        if f.lower().endswith('.wav'):
            REFERENCE_WAV = os.path.join(root, f)
            break
    if REFERENCE_WAV:
        break

info = sf.info(REFERENCE_WAV)
dur  = info.frames / info.samplerate
print(f'\n🎙️  Reference : {REFERENCE_WAV}')
print(f'   Duration  : {dur:.2f}s | SR: {info.samplerate} Hz')
if dur < 3:
    print('   ⚠️  File quá ngắn, lý tưởng 5-10 giây')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 6 — Cấu hình training
# ════════════════════════════════════════════════════════════
from xtts_finetune.config import TrainingConfig

WORKING    = '/kaggle/working'
OUTPUT_DIR = f'{WORKING}/output'

config = TrainingConfig(
    # ── Model ──────────────────────────────────────────────
    hf_repo_id     = 'anhnh2002/vnTTS',
    base_model_dir = f'{WORKING}/base_model',

    # ── Data (dùng manifest đã fix path) ───────────────────
    train_manifest  = FIXED_TRAIN,
    val_manifest    = FIXED_TEST,
    reference_audio = REFERENCE_WAV,
    audio_root_remap = None,   # path đã fix sẵn, không cần remap nữa

    patch_size = 5000,

    # ── Output ─────────────────────────────────────────────
    output_dir     = OUTPUT_DIR,
    checkpoint_dir = f'{OUTPUT_DIR}/checkpoints',
    sample_dir     = f'{OUTPUT_DIR}/samples',
    log_dir        = f'{OUTPUT_DIR}/logs',

    # ── Training (T4 16GB) ─────────────────────────────────
    batch_size       = 2,
    grad_accum_steps = 8,      # effective batch = 16
    learning_rate    = 2e-5,
    epochs_per_patch = 1,

    # ── Memory ─────────────────────────────────────────────
    use_fp16               = True,
    gradient_checkpointing = True,
    freeze_encoder         = True,

    # ── Speaker ────────────────────────────────────────────
    speaker_mode = 'single',

    # ── Kaggle ─────────────────────────────────────────────
    zip_checkpoints = True,
    num_workers     = 2,
    seed            = 42,
)

print('✅ Config sẵn sàng')
print(f'   Train manifest  : {config.train_manifest}')
print(f'   Val manifest    : {config.val_manifest}')
print(f'   Reference wav   : {config.reference_audio}')
print(f'   Effective batch : {config.batch_size * config.grad_accum_steps}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 7 — Load + validate dataset
# ════════════════════════════════════════════════════════════
from xtts_finetune.dataset import load_manifest, validate_and_filter

logger = get_logger('kaggle', config.log_dir)
set_seed(42)

print('📂 Loading train manifest...')
train_raw = load_manifest(config.train_manifest, logger)

print('📂 Loading val manifest...')
val_raw = load_manifest(config.val_manifest, logger)

# Kiểm tra 3 mẫu đầu
print('\n🔍 Kiểm tra 3 mẫu đầu:')
for s in train_raw[:3]:
    exists = '✅' if os.path.isfile(s['audio']) else '❌'
    print(f'  {exists} {s["audio"]}')
    print(f'     text: "{s["text"][:80]}"')

found = sum(1 for s in train_raw[:100] if os.path.isfile(s['audio']))
print(f'\n📊 100 mẫu đầu: {found}/100 file tồn tại')
if found < 90:
    print('⚠️  Nhiều file không tìm thấy! Kiểm tra lại Cell 5')

print('\n🔍 Validating train...')
train_samples = validate_and_filter(train_raw, config, logger)

print('🔍 Validating val...')
val_samples = validate_and_filter(val_raw, config, logger)

print(f'\n✅ Train : {len(train_samples):,} mẫu hợp lệ')
print(f'✅ Val   : {len(val_samples):,} mẫu hợp lệ')

if len(train_samples) == 0:
    raise RuntimeError('❌ Không có mẫu train hợp lệ!')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 8 — Download base model XTTS từ HuggingFace
# ════════════════════════════════════════════════════════════
from xtts_finetune.model_loader import download_base_model

download_base_model(config, logger)
print('✅ Base model sẵn sàng')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 9 — Load model + cấu hình trainable params
# ════════════════════════════════════════════════════════════
from xtts_finetune.model_loader import (
    load_xtts_model,
    configure_trainable_params,
    enable_gradient_checkpointing,
    extract_speaker_embedding,
)
from xtts_finetune.utils import log_gpu_memory

model, xtts_config = load_xtts_model(config, logger)
model = configure_trainable_params(model, config, logger)
enable_gradient_checkpointing(model, logger)
log_gpu_memory(logger, 'sau khi load model')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
speaker_embedding = extract_speaker_embedding(
    model, xtts_config, config.reference_audio, device, logger
)

if speaker_embedding is not None:
    print(f'✅ Speaker embedding: {speaker_embedding.shape}')
else:
    print('⚠️  Không extract được speaker embedding, dùng default')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 10 — BẮT ĐẦU TRAINING
# ════════════════════════════════════════════════════════════
from xtts_finetune.trainer import XTTSTrainer

trainer = XTTSTrainer(
    model             = model,
    xtts_config       = xtts_config,
    config            = config,
    speaker_embedding = speaker_embedding,
)

# ── RESUME (nếu session bị ngắt giữa chừng) ───────────────────
# Bỏ comment 5 dòng dưới để resume:
# from xtts_finetune.utils import find_latest_checkpoint
# from xtts_finetune.model_loader import load_checkpoint
# ckpt = find_latest_checkpoint(config.checkpoint_dir)
# if ckpt:
#     meta = load_checkpoint(ckpt, model, logger=logger)
#     trainer.global_step = meta['step']
#     config.resume_patch = meta['patch_idx'] + 1
#     print(f'▶️  Resumed từ patch {meta["patch_idx"]}, step {meta["step"]}')

print('🚀 Bắt đầu training...')
final_metrics = trainer.train(train_samples, val_samples)

print(f'\n🎉 Training hoàn tất!')
print(f'   Best val loss : {trainer.best_val_loss:.4f}')
print(f'   Final val MCD : {final_metrics.get("val_mcd", "N/A")}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 11 — Inference: tổng hợp giọng nói từ text
# ════════════════════════════════════════════════════════════
from xtts_finetune.inference import run_inference

test_texts = [
    'Xin chào, đây là hệ thống chuyển văn bản thành giọng nói tiếng Việt.',
    'Hôm nay trời đẹp, tôi rất vui được gặp bạn.',
    'Công nghệ trí tuệ nhân tạo đang phát triển rất nhanh.',
]

for i, text in enumerate(test_texts):
    out_path = f'{OUTPUT_DIR}/test_{i+1:02d}.wav'
    run_inference(
        text            = text,
        reference_audio = config.reference_audio,
        output_path     = out_path,
        config          = config,
        logger          = logger,
    )
    print(f'✅ [{i+1}] {out_path}')

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 12 — Nghe thử audio ngay trong notebook
# ════════════════════════════════════════════════════════════
from IPython.display import Audio, display

for i, text in enumerate(test_texts):
    path = f'{OUTPUT_DIR}/test_{i+1:02d}.wav'
    if os.path.isfile(path):
        print(f'\n📢 [{i+1}] "{text[:70]}"')
        display(Audio(path))

In [ ]:
# ════════════════════════════════════════════════════════════
# CELL 13 — Zip output để download về máy
# ════════════════════════════════════════════════════════════
import shutil

zip_path = '/kaggle/working/xtts_output.zip'
shutil.make_archive('/kaggle/working/xtts_output', 'zip', OUTPUT_DIR)

size_mb = os.path.getsize(zip_path) / 1024**2
print(f'✅ Đã zip: {zip_path} ({size_mb:.1f} MB)')
print('👉 Download: Kaggle → Output tab → xtts_output.zip')